### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are pairings of: 
1. A schema, including the name of the tool, a description, and/or argument definitions(often a JSON Schema)
2. A function or coroutine to execute.

In [8]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model
model=init_chat_model("groq:qwen/qwen3.8-27b")
response=model.invoke("Why do parrots talk?")
response

AIMessage(content='The short answer is that parrots don’t actually "talk" in the way humans do. They are **imitating** sounds they hear, including human speech, because of their unique vocal anatomy and social nature. Here’s a detailed breakdown of why and how they do it:\n\n### 1. Anatomical Ability: The Syrinx\nParrots have a specialized vocal organ at the base of their trachea called the **syrinx**. Unlike the larynx in humans, the syrinx allows birds to produce a wide range of pitches and complex sounds. Parrots, in particular, have exceptional control over this organ, enabling them to mimic the nuances of human speech with surprising accuracy.\n\n### 2. Social Motivation: Bonding and Communication\nParrots are highly **social and flock-oriented** animals. In the wild, they use vocalizations to:\n- Maintain contact with their flock.\n- Establish territory.\n- Signal danger.\n- Strengthen social bonds.\n\nWhen kept as pets, parrots view their human caregivers as part of their "flock

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at the a location"""
    return f"The weather at {location} is sunny!"

model_with_tools=model.bind_tools([get_weather]) 

In [9]:
response = model_with_tools.invoke("What is the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")



content='' additional_kwargs={'tool_calls': [{'id': '8yjd4nvh5', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 279, 'total_tokens': 305, 'completion_time': 0.068026273, 'completion_tokens_details': None, 'prompt_time': 0.019778608, 'prompt_tokens_details': None, 'queue_time': 0.049960242, 'total_time': 0.087804881}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a18899-dda8-716e-bc6b-998f1e7421d0-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '8yjd4nvh5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 279, 'output_tokens': 26, 'total_tokens': 305}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [11]:
# Step1 : Model generates tool calls
messages=[{"role":"user", "content":"What's the weather in Boston?"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

print(messages)

# Step2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step3 : Pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


[{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jy2b57ksq', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.068197045, 'completion_tokens_details': None, 'prompt_time': 0.020854301, 'prompt_tokens_details': None, 'queue_time': 0.053504189, 'total_time': 0.089051346}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a188a2-dde4-70d2-a019-5fa1f0344ab4-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'jy2b57ksq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304})]
The weather in Boston is sunny! ☀️

In [12]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jy2b57ksq', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.068197045, 'completion_tokens_details': None, 'prompt_time': 0.020854301, 'prompt_tokens_details': None, 'queue_time': 0.053504189, 'total_time': 0.089051346}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a188a2-dde4-70d2-a019-5fa1f0344ab4-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'jy2b57ksq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}),
 ToolMessage(content='The weather